# 00 - Diagnostico, alcance e ingesta reproducible

Este notebook es el punto de entrada del proyecto. Su objetivo es construir una version canonica y reproducible del dataset ECtHR Task B para que todo el analisis posterior use exactamente los mismos casos, textos, etiquetas y particiones.

El problema se formula como un sistema de apoyo al analisis inicial de casos ante el Tribunal Europeo de Derechos Humanos. Dado el texto de los hechos, el sistema debe ayudar a tres tareas practicas: sugerir articulos potencialmente implicados, recuperar precedentes similares y producir explicaciones auxiliares que permitan auditar por que el modelo ha activado ciertas etiquetas.

La idea de principio a fin es:

```text
caso juridico en texto
        ↓
normalizacion y union de parrafos
        ↓
tabla de casos + tabla de etiquetas
        ↓
representacion numerica en los notebooks siguientes
        ↓
modelos, retrieval, explicabilidad y drift
```

En este primer cuaderno todavia no se entrena ningun modelo. Aqui se fijan las bases del experimento: que entra, como se guarda y que significa cada tabla.


In [1]:
from pathlib import Path
import json, sqlite3, re, random, math, warnings, sys, subprocess, importlib.util
from datetime import datetime, timezone
from uuid import uuid4
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd().resolve()
DATA = ROOT / 'data'
RAW = DATA / 'raw'
INTERIM = DATA / 'interim'
PROCESSED = DATA / 'processed'
ARTIFACTS = ROOT / 'artifacts'
FIGURES = ARTIFACTS / 'figures'
METRICS = ARTIFACTS / 'metrics'
MODELS = ARTIFACTS / 'models'
INDICES = ARTIFACTS / 'indices'
REPORTS = ARTIFACTS / 'reports'
DB = INTERIM / 'metadata.db'
SCHEMA_PATH = ROOT / 'schema.sql'
for d in [RAW, INTERIM, PROCESSED, FIGURES, METRICS, MODELS, INDICES, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package = pip_name or import_name
        print(f'Instalando {package} en este kernel...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

def dump_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

print('ROOT =', ROOT)


ROOT = C:\Users\jordi\Documents\UNI\proyecto-aprendizaje-avanzado


## Ejemplo minimo: de parrafos juridicos a una fila tabular

Un caso del TEDH puede venir segmentado en varios parrafos o fragmentos. Para el modelo necesitamos una unidad documental completa: `text_full`. Por eso se unen los fragmentos preservando el orden y se calculan metadatos basicos.

El ejemplo siguiente usa un caso ficticio pequeno para mostrar exactamente que se guarda: identificador, texto unido, numero de parrafos, numero aproximado de tokens y etiquetas asociadas.


In [2]:
toy_case = {
    'case_id': 'toy_case_001',
    'split': 'train',
    'year': 2018,
    'paragraphs': [
        'The applicant complained about unlawful detention.',
        'He also alleged that the domestic proceedings were not fair.'
    ],
    'article_ids': ['5', '6']
}

toy_text_full = ' '.join(toy_case['paragraphs'])
toy_cases = pd.DataFrame([{
    'case_id': toy_case['case_id'],
    'split': toy_case['split'],
    'year': toy_case['year'],
    'text_full': toy_text_full,
    'n_paragraphs': len(toy_case['paragraphs']),
    'n_tokens': len(toy_text_full.split()),
}])
toy_case_labels = pd.DataFrame([
    {'case_id': toy_case['case_id'], 'article_id': art, 'value': 1}
    for art in toy_case['article_ids']
])

print('Tabla cases:')
display(toy_cases)
print('Tabla case_labels:')
display(toy_case_labels)


Tabla cases:


,case_id,split,year,text_full,n_paragraphs,n_tokens
0,toy_case_001,train,2018,The applicant complained about unlawful detent...,2,16


Tabla case_labels:


,case_id,article_id,value
0,toy_case_001,5,1
1,toy_case_001,6,1


## Esquema de datos

Aunque ECtHR Task B es un dataset textual, lo organizamos como un conjunto de tablas auditables. Esta estructura permite separar cuatro conceptos que no deben mezclarse:

- `cases`: una fila por caso, con `case_id`, `split`, `year`, `text_full`, `n_paragraphs` y `n_tokens`.
- `case_paragraphs`: los fragmentos originales del caso, utiles para inspeccion y futuras extensiones por parrafo.
- `articles`: el catalogo de etiquetas posibles.
- `case_labels`: la relacion multietiqueta entre casos y articulos.
- `predictions`, `experiment_runs` y `explanations`: resultados generados por los notebooks posteriores.

Separar datos, etiquetas y predicciones ayuda a evitar errores metodologicos. Por ejemplo, un caso de `test` puede tener etiquetas verdaderas para evaluar, pero esas etiquetas no deben usarse para entrenar ni para ajustar umbrales.


In [3]:
def get_conn():
    conn = sqlite3.connect(DB)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    conn = get_conn()
    conn.executescript(SCHEMA_PATH.read_text(encoding='utf-8'))
    conn.commit()
    return conn


## Ejemplo minimo: matriz multietiqueta `Y`

Como un caso puede activar varios articulos, la salida no es una unica clase. Se construye una matriz binaria `Y` con una fila por caso y una columna por articulo. Un `1` significa que el articulo aparece asociado al caso; un `0` significa que no.


In [4]:
toy_labels_long = pd.DataFrame([
    {'case_id': 'case_A', 'article_id': '5', 'value': 1},
    {'case_id': 'case_A', 'article_id': '6', 'value': 1},
    {'case_id': 'case_B', 'article_id': '8', 'value': 1},
    {'case_id': 'case_C', 'article_id': '6', 'value': 1},
])

toy_Y = toy_labels_long.pivot_table(
    index='case_id', columns='article_id', values='value', fill_value=0, aggfunc='max'
).astype(int)

display(toy_Y)
print('Y.shape =', toy_Y.shape, '= numero de casos x numero de articulos')


article_id,5,6,8
case_id,,,
case_A,1,1,0
case_B,0,0,1
case_C,0,1,0


Y.shape = (3, 3) = numero de casos x numero de articulos


## Funciones de normalizacion

Las funciones siguientes convierten entradas heterogeneas en una representacion estable. La normalizacion hace cuatro tareas:

1. Localiza el campo textual aunque el dataset lo nombre de forma distinta.
2. Une segmentos o parrafos en `text_full` sin cambiar el orden.
3. Extrae metadatos simples como longitud y numero de parrafos.
4. Mantiene un identificador estable para poder conectar casos, etiquetas, predicciones y explicaciones.

No se elimina informacion juridica de forma agresiva. En textos legales, palabras aparentemente repetidas o formulas estandar pueden ser senales utiles para ciertos articulos; por eso la limpieza se mantiene conservadora.


In [5]:
TEXT_KEYS = ('text', 'facts', 'fact', 'document', 'content')
ID_KEYS = ('case_id', 'id', 'doc_id')
YEAR_KEYS = ('year', 'decision_year', 'judgment_year', 'date', 'decision_date', 'judgment_date')

def extract_text_segments(example):
    text_value = ''
    for key in TEXT_KEYS:
        if key in example and example[key] is not None:
            text_value = example[key]
            break
    if isinstance(text_value, str):
        cleaned = text_value.strip()
        return [cleaned] if cleaned else []
    if isinstance(text_value, list):
        return [str(x).strip() for x in text_value if isinstance(x, str) and str(x).strip()]
    return []

def parse_year(value):
    if isinstance(value, bool):
        return None
    if isinstance(value, int) and 1900 <= value <= 2100:
        return value
    if isinstance(value, float) and value.is_integer() and 1900 <= int(value) <= 2100:
        return int(value)
    if isinstance(value, str):
        m = re.search(r'(19|20)\d{2}', value)
        if m:
            return int(m.group(0))
    return None

def extract_year(example):
    for key in YEAR_KEYS:
        if key in example and example[key] is not None:
            y = parse_year(example[key])
            if y is not None:
                return y
    return None

def extract_label_ids(raw_labels, label_count=None):
    if isinstance(raw_labels, (int, float, bool)):
        v = int(raw_labels)
        return [v] if v >= 0 else []
    if not isinstance(raw_labels, (list, tuple)) or len(raw_labels) == 0:
        return []
    values = [int(v) for v in raw_labels if isinstance(v, (int, float, bool))]
    if label_count is not None and len(values) == label_count and set(values).issubset({0, 1}):
        return [i for i, v in enumerate(values) if v == 1]
    return sorted({v for v in values if v >= 0})

def build_case_id(example, task, split, row_idx):
    for key in ID_KEYS:
        if key in example and example[key] is not None:
            safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(example[key])).strip('_')
            if safe:
                return f'{task}_{split}_{safe}'
    return f'{task}_{split}_{row_idx:06d}'

def build_case_record(example, task, split, row_idx):
    case_id = build_case_id(example, task, split, row_idx)
    segments = extract_text_segments(example)
    text_full = '\n\n'.join(segments)
    n_tokens = len(re.findall(r'\w+', text_full, flags=re.UNICODE))
    return {
        'case_id': case_id,
        'task': task,
        'split': split,
        'year': extract_year(example),
        'text_full': text_full,
        'n_paragraphs': len(segments),
        'n_tokens': n_tokens,
    }, segments


## Carga de LexGLUE ECtHR Task B

La tarea B de ECtHR es una clasificacion multietiqueta: un mismo caso puede estar asociado a varios articulos. En terminos de ML, cada caso se transforma en un vector binario `y` donde cada posicion indica si un articulo esta presente o no.

El notebook usa Hugging Face Datasets con cache local en `data/raw/hf_cache`. Para el informe y la defensa no se debe ejecutar con muestras diminutas: las metricas validas salen de la ingesta completa. Si se limita el numero de filas solo sirve como prueba tecnica, no como resultado academico.


In [6]:
ensure_package('datasets')
from datasets import load_dataset

TASK = 'ecthr_task_b'
DATASET_NAME = 'lex_glue'
DATASET_SUBSET = 'ecthr_b'
CACHE_DIR = RAW / 'hf_cache'

dataset = load_dataset(DATASET_NAME, DATASET_SUBSET, cache_dir=str(CACHE_DIR))
print(dataset)


DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 9000
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 1000
    })
})


In [7]:
def extract_label_names(dataset_dict):
    split_name = 'train' if 'train' in dataset_dict else list(dataset_dict.keys())[0]
    features = getattr(dataset_dict[split_name], 'features', None)
    if not features or 'labels' not in features:
        return []
    label_feature = features['labels']
    if hasattr(label_feature, 'feature') and hasattr(label_feature.feature, 'names'):
        return [str(x) for x in label_feature.feature.names]
    if hasattr(label_feature, 'names'):
        return [str(x) for x in label_feature.names]
    return []

label_names = extract_label_names(dataset)
label_count = len(label_names) if label_names else None
print('Labels:', label_names)


Labels: ['2', '3', '5', '6', '8', '9', '10', '11', '14', 'P1-1']


## Ingesta completa en SQLite

Esta celda materializa la version canonica del dataset usada por los demas notebooks. El resultado principal es `data/interim/metadata.db`, junto con un JSON de estado en `artifacts/reports/ingestion_status.json`.

El objetivo de guardar SQLite no es introducir una dependencia externa, sino fijar una frontera reproducible: todos los notebooks posteriores leen la misma tabla de casos y etiquetas. Asi se evita que cada experimento vuelva a interpretar el dataset de forma distinta.


In [8]:
conn = init_db()
case_rows, paragraph_rows, label_rows = [], [], []
discovered_labels = set()
split_counts = {}

for split_name in dataset.keys():
    split_counts[split_name] = 0
    for row_idx, example in enumerate(dataset[split_name]):
        case_record, segments = build_case_record(example, task=TASK, split=split_name, row_idx=row_idx)
        case_rows.append(case_record)
        for pidx, paragraph in enumerate(segments):
            paragraph_rows.append({'case_id': case_record['case_id'], 'paragraph_idx': pidx, 'paragraph_text': paragraph})
        labels = extract_label_ids(example.get('labels'), label_count=label_count)
        for label_id in labels:
            discovered_labels.add(label_id)
            label_rows.append({'case_id': case_record['case_id'], 'article_id': str(label_id), 'value': 1})
        split_counts[split_name] += 1

if label_names:
    article_rows = [{'article_id': str(i), 'article_code': name, 'description': ''} for i, name in enumerate(label_names)]
else:
    article_rows = [{'article_id': str(i), 'article_code': f'article_{i}', 'description': ''} for i in sorted(discovered_labels)]

with conn:
    conn.executemany('INSERT OR REPLACE INTO cases VALUES (:case_id, :task, :split, :year, :text_full, :n_paragraphs, :n_tokens)', case_rows)
    conn.executemany('INSERT OR REPLACE INTO case_paragraphs VALUES (:case_id, :paragraph_idx, :paragraph_text)', paragraph_rows)
    conn.executemany('INSERT OR REPLACE INTO articles VALUES (:article_id, :article_code, :description)', article_rows)
    conn.executemany('INSERT OR REPLACE INTO case_labels VALUES (:case_id, :article_id, :value)', label_rows)
    counts = {t: conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0] for t in ['cases','case_paragraphs','articles','case_labels','experiment_runs','predictions','explanations']}
conn.close()

report = {
    'stage': 'ingestion_notebook',
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'dataset': {'name': DATASET_NAME, 'subset': DATASET_SUBSET, 'cache_dir': str(CACHE_DIR)},
    'task': TASK,
    'limit_per_split': None,
    'split_counts': split_counts,
    'inserted': {'articles': len(article_rows), 'cases': len(case_rows), 'case_paragraphs': len(paragraph_rows), 'case_labels': len(label_rows)},
    'table_counts': counts,
    'sqlite_path': str(DB),
}
dump_json(report, REPORTS / 'ingestion_status.json')
print(report)
assert counts['cases'] == 11000, 'La ingesta debe contener 11000 casos.'


{'stage': 'ingestion_notebook', 'timestamp_utc': '2026-05-05T09:14:39.235620+00:00', 'dataset': {'name': 'lex_glue', 'subset': 'ecthr_b', 'cache_dir': 'C:\\Users\\jordi\\Documents\\UNI\\proyecto-aprendizaje-avanzado\\data\\raw\\hf_cache'}, 'task': 'ecthr_task_b', 'limit_per_split': None, 'split_counts': {'train': 9000, 'test': 1000, 'validation': 1000}, 'inserted': {'articles': 10, 'cases': 11000, 'case_paragraphs': 262580, 'case_labels': 15991}, 'table_counts': {'cases': 11000, 'case_paragraphs': 262580, 'articles': 10, 'case_labels': 15991, 'experiment_runs': 0, 'predictions': 0, 'explanations': 0}, 'sqlite_path': 'C:\\Users\\jordi\\Documents\\UNI\\proyecto-aprendizaje-avanzado\\data\\interim\\metadata.db'}


In [9]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import json, re, sqlite3
from pathlib import Path

COLORS = {
    'navy': '#334155',
    'teal': '#8dd3c7',
    'blue': '#80b1d3',
    'coral': '#fb8072',
    'amber': '#fdb462',
    'purple': '#bebada',
    'gray': '#64748b',
    'lightgray': '#d9d9d9',
    'offwhite': '#f8fafc',
}

def paper_style():
    sns.set_palette('Set3')
    plt.rcParams.update({
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'axes.edgecolor': COLORS['navy'],
        'axes.labelcolor': COLORS['navy'],
        'axes.titlecolor': COLORS['navy'],
        'axes.titlesize': 11,
        'axes.labelsize': 9,
        'xtick.color': COLORS['navy'],
        'ytick.color': COLORS['navy'],
        'xtick.labelsize': 8,
        'ytick.labelsize': 8,
        'font.size': 9,
        'legend.fontsize': 8,
        'savefig.facecolor': 'white',
        'savefig.bbox': 'tight',
    })

def save_paper_figure(fig, name):
    FIGURES.mkdir(parents=True, exist_ok=True)
    out = []
    for ext in ['pdf', 'png']:
        p = FIGURES / f'{name}.{ext}'
        fig.savefig(p, dpi=240 if ext == 'png' else None, bbox_inches='tight')
        out.append(p)
    plt.close(fig)
    return out

def box(ax, xy, w, h, text, face='white', edge=None, fontsize=9, weight='normal', color=None):
    patch = FancyBboxPatch(
        xy, w, h,
        boxstyle='round,pad=0.018,rounding_size=0.018',
        linewidth=1.15,
        edgecolor=edge or COLORS['navy'],
        facecolor=face,
    )
    ax.add_patch(patch)
    ax.text(
        xy[0] + w/2, xy[1] + h/2, text,
        ha='center', va='center',
        fontsize=fontsize,
        color=color or COLORS['navy'],
        fontweight=weight,
        linespacing=1.12,
    )
    return patch

def arrow(ax, start, end, color=None, rad=0.0, lw=1.5):
    patch = FancyArrowPatch(
        start, end,
        arrowstyle='-|>',
        mutation_scale=13,
        linewidth=lw,
        color=color or COLORS['gray'],
        connectionstyle=f'arc3,rad={rad}',
    )
    ax.add_patch(patch)
    return patch

def load_project_frames():
    conn = sqlite3.connect(DB)
    cases_df = pd.read_sql_query(
        'SELECT case_id, split, year, text_full, n_paragraphs, n_tokens FROM cases',
        conn,
    )
    labels_df = pd.read_sql_query(
        'SELECT case_id, article_id, value FROM case_labels WHERE value = 1',
        conn,
    )
    articles_df = pd.read_sql_query(
        'SELECT article_id, article_code, description FROM articles ORDER BY CAST(article_id AS INTEGER)',
        conn,
    )
    conn.close()
    labels_df['article_id'] = labels_df['article_id'].astype(str)
    articles_df['article_id'] = articles_df['article_id'].astype(str)
    return cases_df, labels_df, articles_df

paper_style()

# Figura 1: overview limpio del sistema.
fig, ax = plt.subplots(figsize=(10.8, 5.8))
ax.axis('off')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

left_x = 0.08
steps = [
    ('Caso jurídico\nbruto', 0.77, COLORS['offwhite']),
    ('Normalización\n+ SQLite', 0.59, '#ecfeff'),
    ('TF-IDF\nunigramas + bigramas', 0.41, '#eef2ff'),
    ('One-vs-Rest\nmultietiqueta', 0.23, '#fff7ed'),
]
for text, y, face in steps:
    box(ax, (left_x, y), 0.24, 0.105, text, face, fontsize=9, weight='bold')
for (_, y0, _), (_, y1, _) in zip(steps[:-1], steps[1:]):
    arrow(ax, (left_x + 0.12, y0), (left_x + 0.12, y1 + 0.105), COLORS['teal'])

box(ax, (0.43, 0.40), 0.22, 0.12, 'Artículos\ncandidatos', '#eff6ff', COLORS['blue'], weight='bold')
arrow(ax, (left_x + 0.24, 0.285), (0.43, 0.46), COLORS['blue'])
ax.text(0.35, 0.35, 'scores +\numbrales', ha='center', va='center', color=COLORS['blue'], fontsize=8)

box(ax, (0.74, 0.61), 0.20, 0.11, 'Retrieval de\nprecedentes', '#f0fdf4', COLORS['teal'], weight='bold')
box(ax, (0.74, 0.38), 0.20, 0.11, 'XAI\nexplicaciones', '#fff7ed', COLORS['amber'], weight='bold')
box(ax, (0.74, 0.16), 0.20, 0.11, 'Drift +\nerrores', '#faf5ff', COLORS['purple'], weight='bold')
arrow(ax, (0.65, 0.48), (0.74, 0.66), COLORS['teal'], rad=0.08)
arrow(ax, (0.65, 0.46), (0.74, 0.435), COLORS['amber'])
arrow(ax, (0.65, 0.43), (0.74, 0.215), COLORS['purple'], rad=-0.08)

box(
    ax, (0.37, 0.06), 0.48, 0.10,
    'Uso previsto: priorización revisable para el analista jurídico.\nNo decide sentencias ni sustituye criterio profesional.',
    '#fff1f2', COLORS['coral'], fontsize=8.8, weight='bold'
)
arrow(ax, (0.54, 0.40), (0.58, 0.16), COLORS['coral'])
ax.text(0.03, 0.95, 'Figura 1. Overview del sistema', fontsize=13, color=COLORS['navy'], fontweight='bold')
written = save_paper_figure(fig, 'fig01_system_overview')

# Figura 2: esquema compacto de transición de estados, estilo paper/Destill.
fig, ax = plt.subplots(figsize=(9.8, 3.8))
ax.axis('off')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
xs = [0.18, 0.50, 0.82]
heads = ['Estado inicial', 'Estado intermedio', 'Estado final']
texts = [
    'Caso ECtHR\ntexto largo',
    'SQLite + TF-IDF\nscores por artículo',
    'Artículos candidatos\nprecedentes + XAI',
]
faces = [COLORS['lightgray'], COLORS['teal'], COLORS['blue']]
for x, h, t, fc in zip(xs, heads, texts, faces):
    ax.text(x, 0.83, h, ha='center', va='center', fontsize=10, color=COLORS['navy'], fontweight='bold')
    box(ax, (x-0.12, 0.43), 0.24, 0.20, t, face=fc, edge=COLORS['navy'], fontsize=9, weight='bold')
for x0, x1 in [(xs[0], xs[1]), (xs[1], xs[2])]:
    arrow(ax, (x0+0.13, 0.53), (x1-0.13, 0.53), COLORS['gray'], lw=1.6)
ax.text(0.5, 0.22, 'Salida revisable para el analista jurídico; no decide sentencias.', ha='center', va='center', fontsize=9, color=COLORS['navy'])
ax.text(0.03, 0.94, 'Figura 2. Estados del flujo', fontsize=12, color=COLORS['navy'], fontweight='bold')
written += save_paper_figure(fig, 'fig02_state_transition_pipeline')

# Figura 8: mapa compacto de contribuciones, sin infografía extensa.
fig, ax = plt.subplots(figsize=(9.8, 4.1))
ax.axis('off')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
box(ax, (0.38, 0.42), 0.24, 0.16, 'Sistema de apoyo\nTEDH Task B', face='white', edge=COLORS['navy'], fontsize=10, weight='bold')
nodes = [
    ('Datos\n11.000 casos', (0.08, 0.70), COLORS['teal']),
    ('Modelo\nmultietiqueta', (0.38, 0.74), COLORS['blue']),
    ('Retrieval\nprecedentes', (0.68, 0.70), COLORS['amber']),
    ('XAI\nglobal/local', (0.68, 0.22), COLORS['purple']),
    ('Drift +\nerrores', (0.38, 0.18), COLORS['coral']),
    ('Discusión\njurídica', (0.08, 0.22), COLORS['lightgray']),
]
center=(0.50,0.50)
for text,(x,y),color in nodes:
    box(ax, (x, y), 0.20, 0.11, text, face=color, edge=COLORS['navy'], fontsize=8.8, weight='bold')
    ax.plot([center[0], x+0.10], [center[1], y+0.055], color=COLORS['gray'], linewidth=1.1, alpha=0.8)
ax.text(0.5, 0.07, 'Contribución: métricas reproducibles conectadas con consecuencias prácticas.', ha='center', va='center', fontsize=8.5, color=COLORS['gray'])
ax.text(0.03, 0.94, 'Figura 8. Mapa de contribuciones', fontsize=12, color=COLORS['navy'], fontweight='bold')
written += save_paper_figure(fig, 'fig08_contributions_summary')

# Prompts y trazabilidad de Image Gen para material conceptual suplementario.
generated = FIGURES / 'generated'
generated.mkdir(parents=True, exist_ok=True)
prompt_file = generated / 'imagegen_prompts.md'
prompt_file.write_text(
    '# Prompts para ilustraciones conceptuales con Image Gen\n\n'
    'Las ilustraciones conceptuales se generaron con la herramienta integrada Image Gen. '
    'No sustituyen a las figuras científicas de Matplotlib y se usan solo como material suplementario.\n\n'
    '## Prompt 1\nCreate a clean academic article illustration showing a legal case text flowing into TF-IDF vectors, multilabel article predictions, precedent retrieval, and explainability. White background, minimal geometric style, navy/teal/amber palette, no logos, Spanish labels.\n\n'
    '## Prompt 2\nCreate a state-transition diagram inspired by puzzle illustrations: initial state = raw ECtHR case, middle state = structured SQLite/TF-IDF/scores, final state = candidate articles + precedents + explanation. Minimal academic style, clear arrows, Spanish labels.\n\n'
    '## Prompt 3\nCreate an editorial figure explaining false positives and false negatives in legal issue spotting: false negative hides a relevant legal article, false positive adds extra research burden. Minimal legal/ML visual language, Spanish labels.\n\n'
    '## Prompt 4\nCreate a conceptual illustration of temporal drift in legal ML: old case distribution shifts into newer case distribution, model performance changes, retraining policy as a decision point. Clean paper-like style, Spanish labels.\n\n'
    '## Prompt 5\nCreate a visual abstract for the paper: ECtHR Task B, 11,000 cases, TF-IDF + One-vs-Rest, retrieval, XAI, drift. Style similar to modern ML research figures, white background, elegant multi-panel layout, no branding.\n',
    encoding='utf-8',
)
print('\n'.join(str(p) for p in written))
print(prompt_file)

C:\Users\jordi\Documents\UNI\proyecto-aprendizaje-avanzado\artifacts\figures\fig01_system_overview.pdf
C:\Users\jordi\Documents\UNI\proyecto-aprendizaje-avanzado\artifacts\figures\fig01_system_overview.png
C:\Users\jordi\Documents\UNI\proyecto-aprendizaje-avanzado\artifacts\figures\fig02_state_transition_pipeline.pdf
C:\Users\jordi\Documents\UNI\proyecto-aprendizaje-avanzado\artifacts\figures\fig02_state_transition_pipeline.png
C:\Users\jordi\Documents\UNI\proyecto-aprendizaje-avanzado\artifacts\figures\fig08_contributions_summary.pdf
C:\Users\jordi\Documents\UNI\proyecto-aprendizaje-avanzado\artifacts\figures\fig08_contributions_summary.png
C:\Users\jordi\Documents\UNI\proyecto-aprendizaje-avanzado\artifacts\figures\generated\imagegen_prompts.md
